# Sprint 3: Query to Retrieval

This notebook validates the Sprint 3 query-to-retrieval flow for the Mini Wikipedia RAG system.

## What this notebook covers
1. Confirm the API server is available
2. Initialize the vector index through document ingestion
3. Submit a user query to the new `/query` endpoint
4. Verify retrieval metadata and returned documents
5. Compare `/query` with the lower-level `/vector-db/search` endpoint

This notebook assumes the server is running locally and Azure authentication is available for real embeddings.

In [1]:
import sys
sys.path.insert(0, '../src')

import json
import requests

BASE_URL = 'http://127.0.0.1:8000'
QUERY_TEXT = 'What is Python?'
TOP_K = 3

print(f'Using server at {BASE_URL}')
print(f'Test query: {QUERY_TEXT}')

Using server at http://127.0.0.1:8000
Test query: What is Python?


## 1. Check Server Readiness

Confirm that the FastAPI app is running before calling Sprint 3 endpoints.

In [2]:
try:
    response = requests.get(f'{BASE_URL}/health')
    print(f'Status code: {response.status_code}')

    if response.status_code == 200:
        print(json.dumps(response.json(), indent=2))
        print('\n✓ API server is reachable')
    else:
        print(f'Error {response.status_code}: {response.text[:200]}')

except requests.exceptions.ConnectionError:
    print('❌ Connection error: Is the server running?')
    print('   Start with: uv run uvicorn api:app --app-dir src --host 127.0.0.1 --port 8000')
except Exception as e:
    print(f'❌ Unexpected error: {type(e).__name__}: {str(e)}')

Status code: 200
{
  "status": "ok"
}

✓ API server is reachable


## 2. Initialize the Vector Index

Load sample documents and build the vector index required for retrieval.

In [8]:
try:
    response = requests.post(f'{BASE_URL}/ingest', params={'document_count': 10})
    print(f'Status code: {response.status_code}')

    if response.status_code == 200:
        ingest_result = response.json()
        print(json.dumps(ingest_result, indent=2))
        print(f'\n✓ Ingested {ingest_result["documents_ingested"]} documents')
        print(f'✓ Index ready: {ingest_result["index_ready"]}')
    else:
        print(f'Error {response.status_code}: {response.text[:200]}')
        print('\n⚠️ The /ingest endpoint requires Azure authentication.')
        print('   Run: azd auth login --scope api://ailab/Model.Access')

except requests.exceptions.ConnectionError:
    print('❌ Connection error: Is the server running?')
except Exception as e:
    print(f'❌ Unexpected error: {type(e).__name__}: {str(e)}')

Status code: 200
{
  "status": "success",
  "documents_ingested": 10,
  "index_ready": true
}

✓ Ingested 10 documents
✓ Index ready: True


## 3. Submit a User Query

Call the new Sprint 3 `/query` endpoint. This exercises the user-query path: request intake, query embedding, similarity search, and document retrieval.

In [9]:
query_payload = {
    'query': QUERY_TEXT,
    'top_k': TOP_K
}

query_result = None

try:
    response = requests.post(f'{BASE_URL}/query', json=query_payload)
    print(f'Status code: {response.status_code}')

    if response.status_code == 200:
        query_result = response.json()
        print(json.dumps(query_result, indent=2))
        print(f'\n✓ Query accepted: {query_result["query"]}')
        print(f'✓ Embedding dimension: {query_result["query_embedding_dimension"]}')
        print(f'✓ Retrieved documents: {len(query_result["documents"])}')
    elif response.status_code == 404:
        print('❌ The /query endpoint is not available on the running server.')
        print('   This usually means the FastAPI server was started before Sprint 3 code was added.')
        print('   Restart the server with: uv run uvicorn api:app --app-dir src --host 127.0.0.1 --port 8000 --reload')
        print('   Then re-run Cells 6 and 8.')
    else:
        print(f'Error {response.status_code}: {response.text[:300]}')

except requests.exceptions.ConnectionError:
    print('❌ Connection error: Is the server running?')
    print('   Start with: uv run uvicorn api:app --app-dir src --host 127.0.0.1 --port 8000 --reload')
except Exception as e:
    print(f'❌ Unexpected error: {type(e).__name__}: {str(e)}')

Status code: 200
{
  "query": "What is Python?",
  "top_k": 3,
  "query_embedding_dimension": 3072,
  "documents": [
    {
      "text": "Python is a high-level, interpreted programming language created by Guido van Rossum. First released in 1991, Python's design philosophy emphasizes code readability.",
      "score": 0.5340655751240777,
      "metadata": {
        "source": "sample",
        "index": 0,
        "title": "Sample Document 0"
      }
    },
    {
      "text": "Natural language processing is a subfield of linguistics, computer science, and artificial intelligence concerned with the interactions between computers and human language.",
      "score": 0.16032029260330433,
      "metadata": {
        "source": "sample",
        "index": 7,
        "title": "Sample Document 7"
      }
    },
    {
      "text": "Data science is an interdisciplinary field that uses scientific methods, processes, algorithms and systems to extract knowledge and insights from structured and unst

## 4. Validate Sprint 3 Outcomes

Run lightweight checks on the `/query` response to confirm the main Sprint 3 outcomes are satisfied.

In [10]:
try:
    if query_result is None:
        print('⚠️ Run Cell 8 successfully first so query_result is populated.')
    else:
        assert query_result['query'] == QUERY_TEXT
        assert query_result['top_k'] == TOP_K
        assert query_result['query_embedding_dimension'] > 0
        assert len(query_result['documents']) > 0

        first_doc = query_result['documents'][0]
        assert 'text' in first_doc
        assert 'metadata' in first_doc

        print('✓ User query accepted by API')
        print('✓ Query converted to an embedding')
        print('✓ Similarity search executed')
        print('✓ Relevant documents retrieved and returned')
except AssertionError as e:
    print(f'❌ Validation failed: {e}')
except Exception as e:
    print(f'❌ Unexpected error: {type(e).__name__}: {str(e)}')

✓ User query accepted by API
✓ Query converted to an embedding
✓ Similarity search executed
✓ Relevant documents retrieved and returned


## 5. Compare with the Low-Level Search Endpoint

This comparison shows how `/query` wraps the lower-level retrieval behavior exposed by `/vector-db/search`.

In [11]:
search_payload = {
    'query': QUERY_TEXT,
    'top_k': TOP_K
}

try:
    response = requests.post(f'{BASE_URL}/vector-db/search', json=search_payload)
    print(f'Status code: {response.status_code}')

    if response.status_code == 200:
        search_results = response.json()
        print(f'Found {len(search_results)} low-level search results')
        for index, result in enumerate(search_results, start=1):
            print(f'\n{index}. Score: {result.get("score", 0):.3f}')
            print(f'   Text: {result["text"][:100]}...')
            print(f'   Metadata: {result["metadata"]}')
    else:
        print(f'Error {response.status_code}: {response.text[:300]}')

except requests.exceptions.ConnectionError:
    print('❌ Connection error: Is the server running?')
except Exception as e:
    print(f'❌ Unexpected error: {type(e).__name__}: {str(e)}')

Status code: 200
Found 3 low-level search results

1. Score: 0.534
   Text: Python is a high-level, interpreted programming language created by Guido van Rossum. First released...
   Metadata: {'source': 'sample', 'index': 0, 'title': 'Sample Document 0'}

2. Score: 0.160
   Text: Natural language processing is a subfield of linguistics, computer science, and artificial intellige...
   Metadata: {'source': 'sample', 'index': 7, 'title': 'Sample Document 7'}

3. Score: 0.158
   Text: Data science is an interdisciplinary field that uses scientific methods, processes, algorithms and s...
   Metadata: {'source': 'sample', 'index': 3, 'title': 'Sample Document 3'}


## Sprint 3 Outcome Checklist

If the cells above succeeded, Sprint 3 retrieval outcomes are covered:
- `POST /query` accepts a user query
- the server converts the query into an embedding
- the vector DB performs similarity search
- relevant documents are returned to the caller
- the low-level search behavior remains inspectable through `/vector-db/search`

For full automated verification, keep using the pytest suite in parallel with this notebook.